# Chapter 5: Using AI with Manim

Manim has a large API, and remembering every class and method is hard. **opencode** is an AI assistant that runs in your terminal and works just like [gemini.com](https://gemini.com), but with direct access to your files: it can *generate* Manim code and *debug* it for you.

In this chapter we'll learn to:
1. **Generate** pure Manim code with opencode.
2. **Debug** a broken scene with opencode.

In [ ]:
from manim import *
config.media_width = "75%"
config.verbosity = "WARNING"

## 5.1 Generating Manim Code with opencode

Open a terminal in your project and run `opencode`. Give it a clear prompt that names the library/version, describes the scene, and asks for a runnable `Scene` class. For example:

> Using **Manim Community Edition**, write a `Scene` called `PlotParabola` that plots $f(x)=x^2$ on axes from -5 to 5, draws the curve in blue, and animates a yellow dot traveling along it from $x=-3$ to $x=3$.

In [ ]:
%%manim -qm PlotParabola

class PlotParabola(Scene):
    def construct(self):
        axes = Axes(
            x_range=[-5, 5, 1],
            y_range=[0, 10, 2],
            x_length=8,
            y_length=5,
            axis_config={"include_numbers": True},
        ).to_edge(DOWN)

        curve = axes.plot(lambda x: x**2, x_range=[-3, 3], color=BLUE)
        label = MathTex("f(x) = x^2", color=YELLOW).next_to(curve, UP)
        dot = Dot(color=YELLOW).move_to(axes.c2p(-3, (-3) ** 2))

        self.play(Create(axes), run_time=1.5)
        self.play(Create(curve), Write(label), run_time=2)
        self.play(MoveAlongPath(dot, curve, run_time=4, rate_func=linear))
        self.wait(1)
        self.play(FadeOut(axes, curve, label, dot))
        self.wait()

### Prompting tips

- Say **"Manim Community Edition"** to avoid the older ManimGL API.
- Be specific about colors, ranges, and timings.
- Ask opencode to **render** the scene (`manim -qm file.py ClassName`) so you see the result immediately.
- Always read the generated code so you learn the API.

## 5.2 Debugging with opencode

When a scene errors out, just tell opencode: *"This scene raises an error, here's the traceback, fix it."* Because opencode reads your files and logs directly, it can usually pinpoint the bug in one step.

The scene below is **intentionally bugged**. Run it to see the `NameError`.

In [ ]:
%%manim -qm BuggedScene

class BuggedScene(Scene):
    def construct(self):
        circle = Circle(color=BLUE)
        square = Square(color=RED)

        self.play(Create(circle))
        self.wait()

        # Oops! There's a typo below. Can you spot it?
        self.play(Transfrom(circle, square))
        self.wait()

You'll see:

```
NameError: name 'Transfrom' is not defined
```

Ask opencode (or paste into gemini.com):

> This Manim CE scene raises `NameError: name 'Transfrom' is not defined`. Fix it.

It recognizes that `Transfrom` is a misspelling of **`Transform`** and returns the corrected code below.

In [ ]:
%%manim -qm FixedScene

class FixedScene(Scene):
    def construct(self):
        circle = Circle(color=BLUE)
        square = Square(color=RED)

        self.play(Create(circle))
        self.wait()

        # Fixed: 'Transform' was misspelled as 'Transfrom'
        self.play(Transform(circle, square))
        self.wait()

### Common mistakes AI catches fast

- **Typos** in animation names (`Transfrom`, `Fadein`).
- **Wrong import**: `manimlib` (ManimGL) vs `manim` (Community).
- **Using** `Scene` for 3D content (needs `ThreeDScene`).
- **Animating** a Mobject that was never added to the scene.
- **Missing** `self.` before `play` / `wait` / `add`.

Always paste the **full traceback** so the assistant sees the exact failing line.

## Summary

Use opencode (or gemini.com) as a Manim pair-programmer: give a **clear prompt** to generate a scene, and paste the **error traceback** when something breaks. With the boilerplate handled, you can focus on the math and the story of your animations.